In [3]:
import pandas as pd
 
demographic   = pd.read_csv('demographic.csv')
diet          = pd.read_csv('diet.csv')
examination   = pd.read_csv('examination.csv')
labs          = pd.read_csv('labs.csv')
medications   = pd.read_csv('medications.csv', encoding='latin1')
questionnaire = pd.read_csv('questionnaire.csv', encoding='latin1')
 

med_agg = medications.groupby('SEQN').agg(
    num_medications  = ('RXDDRUG', 'count'),   
    on_any_medication = ('RXDUSE', 'first')    
).reset_index()
 

merged = demographic.copy()
 
for df in [diet, examination, labs, med_agg, questionnaire]:
    merged = merged.merge(df, on='SEQN', how='left', suffixes=('', '_dup'))
 
merged.to_csv('nhanes_merged.csv', index=False)
 
print(merged.shape)

(10175, 1814)


In [8]:
import numpy as np
 
df = pd.read_csv('nhanes_merged.csv')

#activity minutes: missing = 0 
activity_minutes = [
    'PAD615',  # vigorous work minutes/day
    'PAD630',  # moderate work minutes/day
    'PAD645',  # walk/bike minutes/day
    'PAD660',  # vigorous recreational minutes/day
    'PAD675',  # moderate recreational minutes/day
]
activity_days = [
    'PAQ610',  # days vigorous work/week
    'PAQ625',  # days moderate work/week
    'PAQ640',  # days walked/biked/week
    'PAQ655',  # days vigorous recreation/week
    'PAQ670',  # days moderate recreation/week
]
df[activity_minutes + activity_days] = df[activity_minutes + activity_days].fillna(0)
 
# total weekly activity minutes
df['total_active_min_per_week'] = (
    (df['PAD615'] * df['PAQ610']) +   # vigorous work
    (df['PAD630'] * df['PAQ625']) +   # moderate work
    (df['PAD645'] * df['PAQ640']) +   # walking/biking
    (df['PAD660'] * df['PAQ655']) +   # vigorous recreation
    (df['PAD675'] * df['PAQ670'])     # moderate recreation
)
 
# Smoking
df['SMD030'] = df['SMD030'].fillna(0)
 
# Alcohol
df['ALQ110'] = df['ALQ110'].fillna(0)
df['ALQ101'] = df['ALQ101'].fillna(0)
 
# Continuous clinical variables
median_cols = [
    'BMXBMI', 'BMXWT', 'BMXHT', 'BMXWAIST',  # body measures
    'LBXTC', 'LBDLDL', 'LBDHDD', 'LBXTR',    # cholesterol
    'LBXSGL', 'LBXIN', 'LBXGH',               # glucose/insulin/HbA1c
    'avg_systolic', 'avg_diastolic',            # blood pressure
    'PAD680',                                   # sedentary minutes
    'INDFMPIR',                                 # income ratio
    'DBD900', 'DBD910',                         # diet
]
for col in median_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
 
# Categorical variables
cat_cols = ['DMDEDUC2', 'SMQ020', 'DIQ010', 'BPQ020', 'on_any_medication']
for col in cat_cols:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
 
df['num_medications'] = df['num_medications'].fillna(0)
 
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
 
print(f"Remaining missing values after cleaning:")
if len(remaining_missing) == 0:
    print("  None! Dataset is fully clean.")
else:
    print(remaining_missing)
 
print(f"\nNew feature added: total_active_min_per_week")
print(f"  Mean:   {df['total_active_min_per_week'].mean():.0f} min/week")
print(f"  Median: {df['total_active_min_per_week'].median():.0f} min/week")
print(f"\nFinal shape: {df.shape}")
 
df.to_csv('nhanes_clean.csv', index=False)
print(f"Saved as nhanes_clean.csv")
 

Remaining missing values after cleaning:
RIDAGEMN    9502
RIDEXMON     362
RIDEXAGM    5962
DMQMILIZ    3914
DMQADFC     9632
            ... 
WHD140      4072
WHQ150      4155
WHQ030M     8697
WHQ500      8697
WHQ520      8697
Length: 1753, dtype: int64

New feature added: total_active_min_per_week
  Mean:   463 min/week
  Median: 60 min/week

Final shape: (10175, 1815)
Saved as nhanes_clean.csv


In [9]:
demographics = [
    'SEQN',        # participant ID
    'RIAGENDR',    # sex (1=male, 2=female)
    'RIDAGEYR',    # age in years
    'RIDRETH3',    # race/ethnicity
    'INDFMPIR',    # poverty-income ratio (socioeconomic status)
    'DMDEDUC2',    # education level
]
 
#physical activity
activity = [
    'PAQ605',  # vigorous work activity (yes/no)
    'PAQ610',  # days of vigorous work activity per week
    'PAD615',  # minutes of vigorous work activity per day
    'PAQ620',  # moderate work activity (yes/no)
    'PAQ625',  # days of moderate work activity per week
    'PAD630',  # minutes of moderate work activity per day
    'PAQ635',  # walk or bicycle for transport (yes/no)
    'PAQ640',  # days walked/biked per week
    'PAD645',  # minutes walked/biked per day
    'PAQ650',  # vigorous recreational activity (yes/no)
    'PAQ655',  # days of vigorous recreational activity per week
    'PAD660',  # minutes of vigorous recreational activity per day
    'PAQ665',  # moderate recreational activity (yes/no)
    'PAQ670',  # days of moderate recreational activity per week
    'PAD675',  # minutes of moderate recreational activity per day
    'PAD680',  # minutes sedentary per day
]
 
body_measures = [
    'BMXWT',      # weight (kg)
    'BMXHT',      # height (cm)
    'BMXBMI',     # BMI
    'BMXWAIST',   # waist circumference (cm)
]
 
lab_results = [
    'LBXGH',    # HbA1c % — key diabetes risk marker 
    'LBXTC',    # total cholesterol
    'LBDLDL',   # LDL cholesterol
    'LBDHDD',   # HDL cholesterol
    'LBXTR',    # triglycerides
    'LBXSGL',   # glucose (non-fasting)
    'LBXIN',    # insulin
]
 
blood_pressure = [
    'BPXSY1',   # systolic BP reading 1
    'BPXDI1',   # diastolic BP reading 1
    'BPXSY2',   # systolic BP reading 2
    'BPXDI2',   # diastolic BP reading 2
]
 
lifestyle = [
    'SMQ020',   # smoked at least 100 cigarettes in life (yes/no)
    'SMD030',   # age started smoking
    'ALQ101',   # had at least 12 drinks in past year (yes/no)
    'ALQ110',   # had at least 12 drinks in life (yes/no)
    'DBD900',   # how many meals from fast food per week
    'DBD910',   # how many ready-to-eat foods per week
]
 
medications = [
    'num_medications',    # total number of medications (engineered)
    'on_any_medication',  # whether on any medication (engineered)
]
 
# self-reported diagnosis 
self_reported = [
    'DIQ010',   # told by doctor they have diabetes (yes/no)
    'BPQ020',   # told by doctor they have high blood pressure (yes/no)
]
 
all_cols = demographics + activity + body_measures + lab_results + blood_pressure + lifestyle + medications + self_reported
df_selected = df[all_cols].copy()

In [10]:
#average the two BP readings 
df_selected['avg_systolic']  = df_selected[['BPXSY1', 'BPXSY2']].mean(axis=1)
df_selected['avg_diastolic'] = df_selected[['BPXDI1', 'BPXDI2']].mean(axis=1)
 
#prediabetes risk flag: HbA1c >= 5.7% (ADA threshold)
df_selected['prediabetes_risk'] = (df_selected['LBXGH'] >= 5.7).astype(int)
 
#hypertension risk flag: systolic >= 130 OR diastolic >= 80 (ACC/AHA threshold)
df_selected['hypertension_risk'] = (
    (df_selected['avg_systolic'] >= 130) |
    (df_selected['avg_diastolic'] >= 80)
).astype(int)
 
#combined target: at risk for either condition
df_selected['high_risk'] = (
    (df_selected['prediabetes_risk'] == 1) |
    (df_selected['hypertension_risk'] == 1)
).astype(int)
 
print(f"Dataset shape: {df_selected.shape}")
print(f"\nTarget variable breakdown:")
print(f"  Prediabetes risk (HbA1c >= 5.7%):        {df_selected['prediabetes_risk'].sum():,} ({df_selected['prediabetes_risk'].mean():.1%})")
print(f"  Hypertension risk (BP >= 130/80):         {df_selected['hypertension_risk'].sum():,} ({df_selected['hypertension_risk'].mean():.1%})")
print(f"  High risk (either condition):             {df_selected['high_risk'].sum():,} ({df_selected['high_risk'].mean():.1%})")
print(f"\nMissing values (top 10):")
print(df_selected.isnull().sum().sort_values(ascending=False).head(10))

df_selected.to_csv('nhanes_selected.csv', index=False)
print(f"\nSaved as nhanes_selected.csv")

Dataset shape: (10175, 52)

Target variable breakdown:
  Prediabetes risk (HbA1c >= 5.7%):        2,109 (20.7%)
  Hypertension risk (BP >= 130/80):         2,087 (20.5%)
  High risk (either condition):             3,218 (31.6%)

Missing values (top 10):
PAQ665          3030
PAQ650          3028
PAQ635          3028
PAQ605          3027
PAQ620          3027
BPXSY1          3003
BPXDI1          3003
BPXSY2          2766
BPXDI2          2766
avg_systolic    2666
dtype: int64

Saved as nhanes_selected.csv
